In [1]:
import csv
import sys
import os

from datetime import datetime, timedelta
from string import Template

In [16]:
def round_to_5min(dt):
    """
    Rounds a datetime object to the nearest 5-minute interval.
    """
    # Calculate the total minutes from the start of the hour
    total_minutes_from_hour = dt.minute + dt.second / 60 + dt.microsecond / (60 * 10**6)

    # Calculate the remainder when divided by 5
    remainder = total_minutes_from_hour % 5

    # Determine the minutes to add or subtract for rounding
    if remainder < 2.5:  # Round down
        minutes_to_adjust = -remainder
    else:  # Round up
        minutes_to_adjust = 5 - remainder

    # Create a new datetime object by adjusting the minutes and setting seconds/microseconds to zero
    rounded_dt = dt + timedelta(minutes=minutes_to_adjust)
    rounded_dt = rounded_dt.replace(second=0, microsecond=0)

    return rounded_dt

In [18]:
filepath_rain='/Users/sahuang/Documents/DATA/am_samoa/Rainfall/'
rain_filename='vaipito_rainfall.csv'

# Read CSV
rain_dt=[]
rain_vals=[]
date_format = '%m/%d/%y %H:%M'
csv_file=filepath_rain+rain_filename
with open(csv_file, 'r', encoding='utf-8-sig') as csvf:
    csvr = csv.DictReader(csvf)
    for row in csvr:
        #print(row)
        input_dt=row['DateTime']#+' '+row['Rainfall (in)']
        #print(input_dt)
        rain_dt_i=datetime.strptime(input_dt,date_format)
        #print(tg_dt_i)
        rain_dt.append(rain_dt_i)
        rain_vals.append(row['Rainfall (in)'])

In [24]:
##### Query Times
date_format = '%m/%d/%y %H:%M'
results_header='        AREA          |   DATE AND TIME      |   RAIN DATE AND TIME   | RAIN (IN)'
results_template=Template('${AREA}     ${DT}    ${RAIN_DT}         ${RAIN_VALS}')

print(results_header)
filepath='/Users/sahuang/Documents/Notes/CSDA - Umbra 24-25/Tide Gauge/'
times_filename='Capture_Times_LocalTime.csv'
csv_file=filepath+times_filename
aoi_names=['Tafuna Plain','Central Tafuna Plain','Pago Pago',"Aunu'u",'Leone','Ofu','Olosega',"Ta'u"]
for aoi_name in aoi_names:
    with open(csv_file, 'r',encoding='utf-8-sig') as csvf:
        csvr = csv.DictReader(csvf)
        for row in csvr:
            if not row[aoi_name]:
                continue
            else:
                aoi_dt_i=datetime.strptime(row[aoi_name],date_format)
                #idx=rain_dt.index(round_to_5min(aoi_dt_i))
                
                closest_datetime = min(rain_dt, key=lambda x: abs(x - round_to_5min(aoi_dt_i)))
                #print('Closest Datetime ',closest_datetime)
                idx=rain_dt.index(closest_datetime)
                
                if rain_vals[idx]=='':
                    continue
                else:            
                    # format for printing
                    results_dct={}
                    results_dct['AREA']=f"{aoi_name:<20}"
                    results_dct['DT']=aoi_dt_i
                    results_dct['RAIN_DT']=rain_dt[idx]
                    results_dct['RAIN_VALS']=f"{rain_vals[idx]:<5}"
                    print(results_template.substitute(**results_dct))
                    #sys.exit()

        AREA          |   DATE AND TIME      |   RAIN DATE AND TIME   | RAIN (IN)
Tafuna Plain             2024-10-19 10:43:00    2024-10-19 10:45:00         0    
Tafuna Plain             2024-11-14 21:32:00    2024-11-14 21:30:00         0    
Tafuna Plain             2024-11-21 10:37:00    2024-11-21 10:35:00         0    
Tafuna Plain             2024-12-17 23:16:00    2024-12-17 23:15:00         0    
Tafuna Plain             2024-12-18 21:30:00    2024-12-18 21:30:00         0    
Tafuna Plain             2025-01-05 21:45:00    2025-01-05 21:45:00         0    
Tafuna Plain             2025-01-08 22:41:00    2025-01-08 22:40:00         0    
Tafuna Plain             2025-01-10 10:38:00    2025-01-10 10:40:00         0.09 
Tafuna Plain             2025-02-10 09:35:00    2025-02-10 09:35:00         0    
Tafuna Plain             2025-02-14 09:30:00    2025-02-13 11:00:00         0    
Tafuna Plain             2025-02-17 22:58:00    2025-02-18 09:55:00         0    
Tafuna Plain    